In [1]:
import os
from collections import defaultdict
from  mapreduce_framework import mapreduce




## **Level 1: Basics**
**Exercise 1.1: Character Counter**


In [2]:
text = [
    "MapReduce is a programming model",
    "for processing large data sets",
]

In [3]:
def mapper(line):

    for char in line.lower():
        if char.isalpha():
            yield (char, 1)


def reducer(key, values):

    return sum(values)
    

In [4]:
# Execute MapReduce
results = mapreduce(text, mapper, reducer)

print(results)


{'m': 4, 'a': 6, 'p': 3, 'r': 6, 'e': 6, 'd': 3, 'u': 1, 'c': 2, 'i': 3, 's': 5, 'o': 4, 'g': 4, 'n': 2, 'l': 2, 'f': 1, 't': 2}


**Exercise 1.2: Long Words**




In [5]:
def mapper(line):
    words = line.lower().split()
    for word in words:
        # Clean basic punctuation
        word = word.strip('.,!?;:"()[]{}')
        if len(word) > 5:
            yield (word, 1)


def reducer(key, values):
    return sum(values)

In [6]:
# Execute MapReduce
results = mapreduce(text, mapper, reducer)

print(results)

{'mapreduce': 1, 'programming': 1, 'processing': 1}


**Exercise 1.3: Average Sales per Product**

In [7]:
sales = [
    {'product': 'Laptop', 'amount': 1200},
    {'product': 'Mouse', 'amount': 25},
    {'product': 'Laptop', 'amount': 1100},
    {'product': 'Mouse', 'amount': 30},
    {'product': 'Laptop', 'amount': 1250},
    {'product': 'Keyboard', 'amount': 75},
    {'product': 'Keyboard', 'amount': 80},
]

In [8]:
def mapper(sale):
    yield (sale['product'], sale['amount'])


def reducer(product, amounts):
    return round(sum(amounts) / len(amounts), 1)

In [9]:
# Execute MapReduce
results = mapreduce(sales, mapper, reducer)

print(results)

{'Laptop': 1183.3, 'Mouse': 27.5, 'Keyboard': 77.5}


**Exercise 1.4: Temperature Statistics**

In [10]:
temperatures = [
    {'city': 'Medellin', 'temperature': 22, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 14, 'date': '2026-01-01'},
    {'city': 'Medellin', 'temperature': 24, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 28, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 13, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 30, 'date': '2026-01-02'},
    {'city': 'Medellin', 'temperature': 23, 'date': '2026-01-03'},
    {'city': 'Bogota', 'temperature': 15, 'date': '2026-01-03'},
    {'city': 'Cartagena', 'temperature': 32, 'date': '2026-01-01'},
    {'city': 'Cartagena', 'temperature': 33, 'date': '2026-01-02'},
]

In [11]:
def mapper(record):

    yield (record['city'], record['temperature'])


def reducer(city, temperatures):
    return {
        'min': min(temperatures),
        'max': max(temperatures),
        'avg': round(sum(temperatures) / len(temperatures), 1)
    }

In [12]:
# Execute MapReduce
results = mapreduce(temperatures, mapper, reducer)

for ciudad, valores in results.items():
    print(f"{ciudad}: {valores}")

Medellin: {'min': 22, 'max': 24, 'avg': 23.0}
Bogota: {'min': 13, 'max': 15, 'avg': 14.0}
Cali: {'min': 28, 'max': 30, 'avg': 29.0}
Cartagena: {'min': 32, 'max': 33, 'avg': 32.5}


**Exercise 1.5: Count by Category**

In [13]:
sales = [
    {'product': 'Laptop', 'category': 'Electronics', 'amount': 1200},
    {'product': 'Mouse', 'category': 'Electronics', 'amount': 25},
    {'product': 'Desk', 'category': 'Furniture', 'amount': 600},
    {'product': 'Chair', 'category': 'Furniture', 'amount': 350},
    {'product': 'Monitor', 'category': 'Electronics', 'amount': 450},
]

In [14]:
def mapper(sale):
    yield (sale['category'], sale['amount'])

def reducer(category, amounts):
    return {
        'count': len(amounts),
        'total': sum(amounts),
        'avg': round(sum(amounts) / len(amounts), 1)
    }

In [15]:
results = mapreduce(sales, mapper, reducer)
for category, valores in results.items():
    print(f"{category}: {valores}")

Electronics: {'count': 3, 'total': 1675, 'avg': 558.3}
Furniture: {'count': 2, 'total': 950, 'avg': 475.0}


# **Level 2: File Processing**

**Exercise 2.1: Word Length Distribution**

In [16]:
file = 'mapreduce/sample_text.txt'

In [17]:
def read_file(filename):
    """Reads a text file and returns lines."""
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

def mapper(line):
    for word in line.split():
        word = word.strip('.,!?')
        if word:
            yield (len(word), 1)

def reducer(length, counts):
    return sum(counts)

In [18]:
 # Execute MapReduce
lines = read_file(file)
results = mapreduce(lines, mapper, reducer)
items = sorted(results.items(), key=lambda x: x[0])
for key, value in items:
    print(f"{key}: {value} words")


1: 10 words
2: 20 words
3: 33 words
4: 17 words
5: 14 words
6: 13 words
7: 12 words
8: 17 words
9: 8 words
10: 12 words
11: 8 words
12: 7 words
13: 3 words
14: 1 words


**Exercise 2.2: Unique Words per File**

In [19]:
directory = 'mapreduce/'

In [20]:
def read_files(directory):
    """Reads all .txt files from a directory."""
    data = []
    for filename in os.listdir(directory):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        data.append((filename, line))
    return data


def mapper(file_data):
    filename, line = file_data
    for word in line.lower().split():
        word = word.strip('.,!?')
        if word:
            yield (filename, word)

def reducer(filename, words):
    return len(set(words))

In [21]:
data = read_files(directory)

results = mapreduce(data, mapper, reducer)

for key, value in results.items():
    print(f"{key}: {value} words")

sample_bigdata.txt: 79 words
sample_text.txt: 116 words


**Exercise 2.3: Inverted Index**

In [22]:
directory = 'mapreduce/'

In [23]:
def read_files(directory):
    """Reads all .txt files from a directory."""
    data = []
    for filename in os.listdir(directory):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        data.append((filename, line))
    return data

def mapper(file_data):
    filename, line = file_data
    for word in line.lower().split():
        word = word.strip('.,!?')
        if word:
            yield (word, filename)

def reducer(word, filenames):
    return sorted(set(filenames))

In [24]:
data = read_files(directory)

results = mapreduce(data, mapper, reducer)
for key, value in results.items():
    print(f"{key}: {value}")

big: ['sample_bigdata.txt', 'sample_text.txt']
data: ['sample_bigdata.txt', 'sample_text.txt']
refers: ['sample_bigdata.txt']
to: ['sample_bigdata.txt', 'sample_text.txt']
extremely: ['sample_bigdata.txt']
large: ['sample_bigdata.txt', 'sample_text.txt']
sets: ['sample_bigdata.txt', 'sample_text.txt']
that: ['sample_bigdata.txt', 'sample_text.txt']
may: ['sample_bigdata.txt']
be: ['sample_bigdata.txt', 'sample_text.txt']
analyzed: ['sample_bigdata.txt']
computationally: ['sample_bigdata.txt']
reveal: ['sample_bigdata.txt']
patterns: ['sample_bigdata.txt']
trends: ['sample_bigdata.txt']
and: ['sample_bigdata.txt', 'sample_text.txt']
associations: ['sample_bigdata.txt']
the: ['sample_bigdata.txt', 'sample_text.txt']
term: ['sample_bigdata.txt']
is: ['sample_bigdata.txt', 'sample_text.txt']
often: ['sample_bigdata.txt']
used: ['sample_bigdata.txt']
describe: ['sample_bigdata.txt']
high: ['sample_bigdata.txt']
in: ['sample_bigdata.txt', 'sample_text.txt']
volume: ['sample_bigdata.txt']
vel

# **Level 3: Advanced Challenges**

**Exercise 3.1: Top-N Words**

In [25]:
file = 'mapreduce/sample_text.txt'

In [26]:
def read_file(filename):
    """Reads a text file and returns lines."""
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

def mapper(line):
    for word in line.split():
        word = word.strip('.,!?')
        if word:
            yield (word, 1)

def reducer(key, values):
    return sum(values)

def top_n_words(text, n=10):
    results = mapreduce(text, mapper, reducer)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:n]

In [27]:
text = read_file(file)
top_words = top_n_words(text, n=10)
for word, count in top_words:
    print(f"{word}: {count} occurrences")


a: 9 occurrences
of: 8 occurrences
and: 7 occurrences
the: 7 occurrences
to: 4 occurrences
distributed: 4 occurrences
MapReduce: 3 occurrences
large: 3 occurrences
data: 3 occurrences
key: 3 occurrences


**Exercise 3.2: Bigram Analysis**

In [28]:
text = ["the quick brown fox", "the quick red dog"]

In [29]:
def mapper(line):
    words = line.lower().split()
    for i in range(len(words) - 1):
        bigram = f"{words[i]} {words[i+1]}"
        yield (bigram, 1)
        
def reducer(key, values):
    """
    Sums all counts for a word.
    
    Args:
        key: The word
        values: List of counts (all are 1)
    
    Returns:
        Total occurrences of the word
    """
    return sum(values)



In [30]:
results = mapreduce(text, mapper, reducer)
print(results)

{'the quick': 2, 'quick brown': 1, 'brown fox': 1, 'quick red': 1, 'red dog': 1}


**Exercise 3.3: Web Session Analysis**

In [31]:
events = [
    {'user': 'U1', 'page': '/home', 'time': '10:00'},
    {'user': 'U1', 'page': '/products', 'time': '10:02'},
    {'user': 'U2', 'page': '/home', 'time': '10:01'},
    {'user': 'U1', 'page': '/cart', 'time': '10:05'},
    {'user': 'U2', 'page': '/about', 'time': '10:03'},
]


In [32]:
def mapper(event):
    yield (event['user'], event['page'])

def reducer(user, pages):
    return {
        'page_views': len(pages),
        'unique_pages': list(set(pages))
    }

In [33]:
results = mapreduce(events, mapper, reducer)
for user, values in results.items():
    print(f"{user}: {values['page_views']} page views, {len(values['unique_pages'])} unique pages")

U1: 3 page views, 3 unique pages
U2: 2 page views, 2 unique pages


**Exercise 3.4: Anomaly Detector**

In [34]:
import math

In [35]:
readings = [
    {'sensor': 'S1', 'value': 22.5},
    {'sensor': 'S1', 'value': 23.0},
    {'sensor': 'S1', 'value': 99.9},  # ← Anomaly
    {'sensor': 'S2', 'value': 15.0},
    {'sensor': 'S2', 'value': 14.8},
]

In [36]:

def mapper(reading):
    """Extrae (sensor, value) de cada lectura."""
    yield (reading['sensor'], reading['value'])


def reducer(sensor, values):
    avg = sum(values) / len(values)
    variance = sum((v - avg) ** 2 for v in values) / len(values)
    std_dev = math.sqrt(variance)
    
    anomalies = []
    for i, v in enumerate(values):
        others = values[:i] + values[i+1:]
        if len(others) < 2:
            continue
        avg_o = sum(others) / len(others)
        std_o = math.sqrt(sum((x - avg_o) ** 2 for x in others) / len(others))
        if abs(v - avg_o) > 2 * std_o:
            anomalies.append(v)
    
    return {
        'avg': round(avg, 2),
        'std_dev': round(std_dev, 2),
        'anomalies': anomalies
    }

In [37]:
results = mapreduce(readings, mapper, reducer)
print(results)

{'S1': {'avg': 48.47, 'std_dev': 36.37, 'anomalies': [99.9]}, 'S2': {'avg': 14.9, 'std_dev': 0.1, 'anomalies': []}}
